In [ ]:
# 第一步：确认真正分配到 L4，然后挂载 Drive 保存检查点。
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), '需要 L4 GPU'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 第二步：上传本地准备的代码和数据快照（仅本项目，无历史模型大文件）。
from google.colab import files
uploaded = files.upload()
print('上传完成:', list(uploaded))

In [ ]:
# 第三步：校验压缩包、解压，并安装辅助依赖。保留 Colab 原有 CUDA PyTorch。
from pathlib import Path
import hashlib, zipfile, shutil, os, subprocess, sys, json
archive = Path('/content/pusht-diffusion-colab-20261007.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '3574b6fcb32039c7421daa23870a602ee2613d78128807e788236e715c8890aa'
with zipfile.ZipFile(archive) as z:
    z.extractall('/content')
ROOT = Path('/content/pusht-diffusion')
DRIVE = Path('/content/drive/MyDrive/pusht-diffusion/20261007-unet-seed0')
DRIVE.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, DRIVE / archive.name)
(DRIVE / 'bundle-sha256.txt').write_text(hashlib.sha256(archive.read_bytes()).hexdigest() + '\n')
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('SETUP_READY', torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 配置补充：按要求使用 8 个 DataLoader 工作进程。
# 同时更新测速脚本和运行记录，确保预检查与正式训练采用相同设置。
changes = {
  "src/pusht_diffusion/learner/training.py": [
    [
      "num_workers=4,",
      "num_workers=8,  # 8 个工作进程并行读取、解码训练数据。"
    ]
  ],
  "scripts/colab_preflight.py": [
    [
      "num_workers=4,",
      "num_workers=8,"
    ],
    [
      "\"batch_size\": config[\"training\"][\"batch_size\"], \"warmup_steps\": 2",
      "\"batch_size\": config[\"training\"][\"batch_size\"], \"num_workers\": 8, \"warmup_steps\": 2"
    ],
    [
      "\"kind\": \"real_l4_preflight\", \"passed\": False",
      "\"kind\": \"real_l4_preflight\", \"num_workers\": 8, \"passed\": False"
    ]
  ],
  "scripts/colab_train.py": [
    [
      "'deterministic_algorithms': True, 'tf32': False,",
      "'deterministic_algorithms': True, 'tf32': False,\n        'num_workers': 8,"
    ]
  ]
}
expected_hashes = {
  "src/pusht_diffusion/learner/training.py": "129be8081b0c66d2ebc639e665bb995b85e333d94e4aaaa5e2f8f1d021166317",
  "scripts/colab_preflight.py": "99c98ff39b4c67cc2f210f40dbea0e633dd55304604d7ebb016cb7e969808fcd",
  "scripts/colab_train.py": "b99dba8caac0e4966100fe4f881ba94e52a91cd6798549f78897e750093b7d19"
}
for name, replacements in changes.items():
    path = ROOT / name
    if hashlib.sha256(path.read_bytes()).hexdigest() == expected_hashes[name]:
        continue
    source = path.read_text()
    for old, new in replacements:
        assert source.count(old) == 1, (name, old)
        source = source.replace(old, new, 1)
    assert hashlib.sha256(source.encode()).hexdigest() == expected_hashes[name]
    path.write_text(source)
# 4-worker 报告单独保留；最终源码包写入 Drive，方便重连后恢复。
old_report = ROOT / 'runs/l4-preflight/preflight.json'
if old_report.exists():
    shutil.copy2(old_report, DRIVE / 'preflight-workers4.json')
final_archive = DRIVE / 'pusht-diffusion-colab-20261007-workers8.zip'
with zipfile.ZipFile(archive) as original, zipfile.ZipFile(final_archive, 'w', zipfile.ZIP_DEFLATED) as updated:
    for item in original.infolist():
        relative = item.filename.removeprefix('pusht-diffusion/')
        updated.writestr(item.filename, (ROOT / relative).read_bytes() if relative in changes else original.read(item.filename))
(DRIVE / 'workers8-source-manifest.json').write_text(json.dumps(expected_hashes, indent=2))
print('WORKERS_8_READY', hashlib.sha256(final_archive.read_bytes()).hexdigest())

In [ ]:
# 输出辅助：把子进程输出实时送到 notebook，保留 \r 的同一行刷新效果。
import codecs
def run_visible(command):
    decoder = codecs.getincrementaldecoder('utf-8')()
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as process:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            print(decoder.decode(chunk), end='', flush=True)
        print(decoder.decode(b'', final=True), end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# 第四步：8-worker L4 预检查，测试权重不用于正式训练。
run_visible([sys.executable, '-u', 'scripts/colab_preflight.py',
    '--dataset', str(ROOT / 'data/pusht_image'),
    '--output', str(ROOT / 'runs/l4-preflight-workers8'),
    '--mirror', str(DRIVE / 'preflight-workers8')])

In [ ]:
# 查看 8-worker 预检查报告。passed=True 才能正式训练。
report = json.loads((ROOT / 'runs/l4-preflight-workers8/preflight.json').read_text())
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
# 输出辅助：把子进程输出实时送到 notebook，保留 \r 的同一行刷新效果。
import codecs
def run_visible(command):
    decoder = codecs.getincrementaldecoder('utf-8')()
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as process:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            print(decoder.decode(chunk), end='', flush=True)
        print(decoder.decode(b'', final=True), end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# 第五步：正式训练。8 个数据读取进程；预检查未通过时拒绝启动。
# 保持 40,000 步、batch=64、seed=0、float32、500 步预热。
# last.pt 每 1,000 步同步 Drive；每 5,000 步额外保留一个快照。
run_visible([sys.executable, '-u', 'scripts/colab_train.py',
    '--dataset', str(ROOT / 'data/pusht_image'),
    '--output', str(ROOT / 'runs/unet-formal'),
    '--mirror', str(DRIVE / 'formal'),
    '--preflight', str(ROOT / 'runs/l4-preflight-workers8/preflight.json')])

## 断线后如何继续
先重新连接 L4 并挂载 Drive。从 `MyDrive/pusht-diffusion/20261007-unet-seed0/pusht-diffusion-colab-20261007-workers8.zip` 恢复最终代码和数据，安装相同依赖。

正式文件夹是该目录下的 `formal/`。在第五步调用中增加 `--resume` 和 `formal/last.pt` 的完整路径，并使用保留的 `preflight-workers8/preflight.json`。目标仍是**累计 40,000 步**。不要重新运行上传和覆盖已有源码的旧补丁格。

首次正式训练从新模型开始；不要使用预检查权重作为正式断点。`num_workers=8` 是读取数据的工作进程数，与 batch=64 不同。每 5,000 步保留一个候选快照，训练后再做开发场景评测；loss 不是闭环成功率。